# Arid Ecosystem Metatranscriptomes
# Mapping

## Creating a concatenated files with all contigs

In [1]:
%%bash

conda init
source ~/.bashrc
conda activate /xdisk/tfaily/vfreirezapata/env_new/seqkit

if [ -s /xdisk/tfaily/vfreirezapata/metat_2025_final/cluster_95_contigs.fa ]
then
    echo "Removing file"
    rm /xdisk/tfaily/vfreirezapata/metat_2025_final/cluster_95_contigs.fa
fi

cd /xdisk/tfaily/cayalaortiz/arid_metaG/annotation/final_dataset/

for i in galah/cluster_95/*
do
    name=$(echo ${i%.fa} | cut -f3 -d "/")
    cat $i | seqkit replace -p ^ -r ${name}_ >> /xdisk/tfaily/vfreirezapata/metat_2025_final/cluster_95_contigs.fa
done

no change     /home/u1/vfreirezapata/miniforge3/condabin/conda
no change     /home/u1/vfreirezapata/miniforge3/bin/conda
no change     /home/u1/vfreirezapata/miniforge3/bin/conda-env
no change     /home/u1/vfreirezapata/miniforge3/bin/activate
no change     /home/u1/vfreirezapata/miniforge3/bin/deactivate
no change     /home/u1/vfreirezapata/miniforge3/etc/profile.d/conda.sh
no change     /home/u1/vfreirezapata/miniforge3/etc/fish/conf.d/conda.fish
no change     /home/u1/vfreirezapata/miniforge3/shell/condabin/Conda.psm1
no change     /home/u1/vfreirezapata/miniforge3/shell/condabin/conda-hook.ps1
no change     /home/u1/vfreirezapata/miniforge3/lib/python3.10/site-packages/xontrib/conda.xsh
no change     /home/u1/vfreirezapata/miniforge3/etc/profile.d/conda.csh
no change     /home/u1/vfreirezapata/.bashrc
No action taken.


### September 2025
### Bowtie2_2.5.4 script

In [1]:
%%bash
pwd

/xdisk/tfaily/vfreirezapata/metat_2025_final


In [2]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=bowtie2_index
#SBATCH --nodes=1
#SBATCH --ntasks=94 
#SBATCH --time=24:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda init
source ~/.bashrc
conda activate /xdisk/tfaily/vfreirezapata/env_new/bowtie2

bowtie2-build /xdisk/tfaily/vfreirezapata/metat_2025_final/cluster_95_contigs.fa bowtie_results/bins_index 

' >scripts/bowtie_index.slurm

In [3]:
%%bash
sbatch scripts/bowtie_index.slurm

Submitted batch job 17458663


# Deinterlieve

In [1]:
%%bash

echo '#!/bin/bash
#SBATCH --job-name=reformat_int
#SBATCH --nodes=1
#SBATCH --ntasks=94 
#SBATCH --time=5:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda activate /xdisk/tfaily/vfreirezapata/env_new/bbtools

# Setting samplename

SRR=$1

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
READS_DIR="${PROJECT_DIR}/sortmerna_results/mrna_reads"

reformat.sh in=${READS_DIR}/${SRR}.sortmerna.interleaved.fq.gz out1=${READS_DIR}/${SRR}_1.fq.gz out2=${READS_DIR}/${SRR}_2.fq.gz

' > scripts/reformat_interlieve.slurm

In [2]:
%%bash
while read SRR
do
    sbatch scripts/reformat_interlieve.slurm $SRR

done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 16217766
Submitted batch job 16217767
Submitted batch job 16217768
Submitted batch job 16217769
Submitted batch job 16217770
Submitted batch job 16217771
Submitted batch job 16217772
Submitted batch job 16217773
Submitted batch job 16217774
Submitted batch job 16217775
Submitted batch job 16217776
Submitted batch job 16217777
Submitted batch job 16217778
Submitted batch job 16217779
Submitted batch job 16217780
Submitted batch job 16217781
Submitted batch job 16217782
Submitted batch job 16217783
Submitted batch job 16217784
Submitted batch job 16217785
Submitted batch job 16217786
Submitted batch job 16217787
Submitted batch job 16217788
Submitted batch job 16217789
Submitted batch job 16217790
Submitted batch job 16217791
Submitted batch job 16217792
Submitted batch job 16217793
Submitted batch job 16217794
Submitted batch job 16217795
Submitted batch job 16217796
Submitted batch job 16217797
Submitted batch job 16217798
Submitted batch job 16217799
Submitted batc

In [3]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=bowtie2_map
#SBATCH --nodes=1
#SBATCH --ntasks=94 
#SBATCH --time=5:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda init
source ~/.bashrc
conda activate /xdisk/tfaily/vfreirezapata/env_new/bowtie2

# Setting samplename

SRR=$1

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
READS_DIR="${PROJECT_DIR}/sortmerna_results/mrna_reads"
MAPPING_DIR="${PROJECT_DIR}/bowtie_results"

bowtie2 -D 20 -R 3 -N 1 -L 20 -i S,1,0.50 -p 45 \
    -1 ${READS_DIR}/${SRR}_1.fq.gz  \
    -2 ${READS_DIR}/${SRR}_2.fq.gz  \
    -x ${MAPPING_DIR}/bins_index/bins_index \
    -S ${MAPPING_DIR}/${SRR}_mapped.sam

echo "#######  SAM to BAM   ###########"

conda activate /xdisk/tfaily/vfreirezapata/env_new/samtools

samtools view -b ${MAPPING_DIR}/${SRR}_mapped.sam -o ${MAPPING_DIR}/${SRR}_mapped.bam

echo "#######  SORTING   ###########"

samtools sort -n ${MAPPING_DIR}/${SRR}_mapped.bam -o ${MAPPING_DIR}/${SRR}_mapped_sorted.bam

' > scripts/bowtie_map_updated.slurm

#### Submitting script


In [4]:
%%bash
while read SRR
do
    sbatch scripts/bowtie_map_updated.slurm $SRR

done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 16218509
Submitted batch job 16218510
Submitted batch job 16218511
Submitted batch job 16218512
Submitted batch job 16218513
Submitted batch job 16218514
Submitted batch job 16218515
Submitted batch job 16218516
Submitted batch job 16218517
Submitted batch job 16218518
Submitted batch job 16218519
Submitted batch job 16218520
Submitted batch job 16218521
Submitted batch job 16218522
Submitted batch job 16218523
Submitted batch job 16218524
Submitted batch job 16218525
Submitted batch job 16218526
Submitted batch job 16218527
Submitted batch job 16218528
Submitted batch job 16218529
Submitted batch job 16218530
Submitted batch job 16218531
Submitted batch job 16218532
Submitted batch job 16218533
Submitted batch job 16218534
Submitted batch job 16218535
Submitted batch job 16218536
Submitted batch job 16218537
Submitted batch job 16218538
Submitted batch job 16218539
Submitted batch job 16218540
Submitted batch job 16218541
Submitted batch job 16218542
Submitted batc

## Transforming annotations

In [1]:
%%bash
/home/u1/vfreirezapata/gffread/gffread \
    /xdisk/tfaily/cayalaortiz/arid_metaG/annotation/final_dataset/dram_sets/genes_merged.gff -T  \
    -o /xdisk/tfaily/vfreirezapata/metat_2025_final/annotation/genes_final.gtf

## Extracting counts

In [5]:
%%bash
conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/subread

featureCounts -p -t CDS \
    -g transcript_id --countReadPairs \
    -a /xdisk/tfaily/vfreirezapata/metat_2025_final/annotation/genes_final.gtf \
    -T 20 -M \
    -o /xdisk/tfaily/vfreirezapata/metat_2025_final/annotation/rnaseq_counts_final_2025.txt \
    /xdisk/tfaily/vfreirezapata/metat_2025_final/bowtie_results/SRR*_mapped_sorted.bam

no change     /home/u1/vfreirezapata/miniforge3/condabin/conda
no change     /home/u1/vfreirezapata/miniforge3/bin/conda
no change     /home/u1/vfreirezapata/miniforge3/bin/conda-env
no change     /home/u1/vfreirezapata/miniforge3/bin/activate
no change     /home/u1/vfreirezapata/miniforge3/bin/deactivate
no change     /home/u1/vfreirezapata/miniforge3/etc/profile.d/conda.sh
no change     /home/u1/vfreirezapata/miniforge3/etc/fish/conf.d/conda.fish
no change     /home/u1/vfreirezapata/miniforge3/shell/condabin/Conda.psm1
no change     /home/u1/vfreirezapata/miniforge3/shell/condabin/conda-hook.ps1
no change     /home/u1/vfreirezapata/miniforge3/lib/python3.10/site-packages/xontrib/conda.xsh
no change     /home/u1/vfreirezapata/miniforge3/etc/profile.d/conda.csh
no change     /home/u1/vfreirezapata/.bashrc
No action taken.



        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.1.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 56 BAM files                                     ||
||                                                                            ||
||                           SRR31679565_mapped_sorted.bam                    ||
||                           SRR31679567_mapped_sorted.bam                    ||
||                           SRR31679568_mapped_sorted.bam       